## Importação das Bibliotecas

In [56]:
import pandas as pd
import requests 
import sqlalchemy
import psycopg2
import json
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
pd.set_option('display.max_colwidth', None)

## Extração do Staging do Verdana

In [4]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Conectar ao banco PostgreSQL usando SQLAlchemy
url = URL.create(
    drivername="postgresql+psycopg2",
    username="admin_pg",
    password="1nbr@ndsIB",
    host="192.168.50.102",
    port=5432,
    database="db_bi_ti"
)
# Criar a engine de conexão
engine = create_engine(url)
# Ler a tabela
df = pd.read_sql('SELECT * FROM f_chamados_verdanadesk', engine)

## Campo de Tratamento dos dados

In [5]:
df.columns = df.columns.str.strip()
df.columns

Index(['id', 'requerente', 'titulo', 'status', 'tipo', 'origem_abertura',
       'prioridade', 'categoria', 'categoria_completa', 'localizacao',
       'nome_tecnico', 'grupo', 'tempo_chamado', 'data_criacao',
       'ultima_atualizacao', 'tempo_atribuicao', 'tempo_solucao',
       'data_solucao', 'data_fechamento', 'expirado', 'tempo_resposta'],
      dtype='str')

In [59]:
map_cidade = {

    'MORUMBI': 'SÃO PAULO',
    'IBIRAPUERA': 'SÃO PAULO',
    'HIGIENOPOLIS': 'SÃO PAULO',
    'LEBLON': 'RIO DE JANEIRO',
    'BARRA SHOPPING': 'RIO DE JANEIRO',
    'PATIO SAVASSI': 'BELO HORIZONTE',
    'IGUATEMI FORTALEZA': 'FORTALEZA'
}

In [61]:
map_estado = {

    'SÃO PAULO': 'SP',
    'RIO DE JANEIRO': 'RJ',
    'BELO HORIZONTE': 'MG',
    'FORTALEZA': 'CE'
}

In [62]:
map_regiao = {

    'SP': 'SUDESTE',
    'RJ': 'SUDESTE',
    'MG': 'SUDESTE',
    'CE': 'NORDESTE'
}

In [57]:
def limpar_unidade(valor):

    if pd.isnull(valor):
        return np.nan

    valor = re.sub(
        r'^[A-Z]{2,10}[0-9]{0,3}\\s*-\\s*',
        '',
        valor
    )

    return valor.strip()

In [53]:
def classificar_tipo_operacao(valor):

    if pd.isnull(valor):
        return np.nan

    valor = valor.upper()

    if 'OUTLET' in valor:
        return 'OUTLET'

    elif valor.startswith('CD'):
        return 'CENTRO DISTRIBUIÇÃO'

    elif 'HUB' in valor:
        return 'HUB LOGÍSTICO'

    elif 'CORPORATIVO' in valor:
        return 'CORPORATIVO'

    else:
        return 'LOJA'

In [51]:
def extrair_marca(valor):

    if pd.isnull(valor):
        return np.nan

    valor = valor.upper()

    if valor.startswith('VR'):
        return 'VR'

    elif valor.startswith('TH'):
        return 'TOMMY HILFIGER'

    elif valor.startswith('ELLUS'):
        return 'ELLUS'

    elif valor.startswith('RC'):
        return "RICHARD'S"

    elif valor.startswith('SL'):
        return 'SALINAS'

    elif valor.startswith('BS'):
        return 'BRANDS HOUSE'

    elif valor.startswith('HUB'):
        return 'HUB'

    elif valor.startswith('CD'):
        return 'CENTRO DISTRIBUIÇÃO'

    elif valor.startswith('INBRANDS'):
        return 'INBRANDS'

    else:
        return 'NÃO IDENTIFICADO'

In [49]:
valores_invalidos = [
    'NÃO ATRIBUÍDO',
    'NAO ATRIBUIDO',
    'NULL',
    'NONE',
    ''
]

In [30]:
df['data_abertura_merge'] = (
    df['data_criacao']
    .dt.date
)

df['data_fechamento_merge'] = (
    df['data_fechamento']
    .dt.date
)

In [7]:
campos_texto = [
    'status',
    'grupo',
    'prioridade',
    'nome_tecnico'
]

for col in campos_texto:

    df[col] = (
        df[col]
        .astype(str)
        .str.upper()
        .str.strip()
    )

In [8]:

valores_nao_atribuidos = [
    'NÃO ATRIBUÍDO',
    'NAO ATRIBUIDO',
    'NULL',
    'NONE',
    ''
]

for col in campos_texto:

    df[col] = np.where(
        df[col].isin(valores_nao_atribuidos),
        np.nan,
        df[col]
    )

In [9]:
campos_data = [
    'data_criacao',
    'tempo_atribuicao',
    'tempo_resposta',
    'data_solucao',
    'data_fechamento'
]

In [10]:
for col in campos_data:

    df[col] = pd.to_datetime(
        df[col],
        errors='coerce'
    )

C:\Users\gustavo.freitas\AppData\Local\Temp\ipykernel_14444\3007463113.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
C:\Users\gustavo.freitas\AppData\Local\Temp\ipykernel_14444\3007463113.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
C:\Users\gustavo.freitas\AppData\Local\Temp\ipykernel_14444\3007463113.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(


In [11]:
### Tratamento dos Dados de Grupo, fazendo a separação:

# Coluna Area_responsavel
map_area = {
    'TI | BUSINESS INTELLIGENCE N1' : 'TI',
    'TI | E-COMMERCE' : 'TI',
    'TI | GOVERNANÇA' : 'TI',
    'TI | INFRAESTRUTURA' : 'TI',
    'TI | LINE CARTUCHOS' : 'TI',
    'TI | SISTEMAS' : 'TI',
    'TI | SISTEMAS N2' : 'TI',
    'TI | SUPORTE A LOJAS (RETAGUARDA)' : 'TI',
    'TI | SUPORTE A LOJAS N2' : 'TI',
    'TI | SUPORTE LOCAL': 'TI',
    'TI | SUPORTE A LOJAS': 'TI',
    'TI | PROJETOS': 'TI',
    'TI | '
    'ADMINISTRAÇÃO DE PESSOAL': 'RH',
    'REMUNERAÇÃO' : 'RH',
    'DEPARTAMENTO FISCAL' : 'FISCAL',
    'MANUTENÇÃO | HUB' : 'MANUTENÇÃO',
    'MANUTENÇÃO | LOJAS' : 'MANUTENÇÃO',
    'GPP / SEGURANÇA (HUB)' : 'TI',
    'GPP / SEGURANÇA (LOJAS)' : 'TI',
    'CROWN IT - ATUALIZAÇÃO DE VERSÃO LINX' : 'TI',
    'CROWN IT N2' : 'TI'

}

#Coluna Torre_responsavel
map_torre = {
    'TI | SUPORTE LOCAL': 'INFRAESTRUTURA',
    'TI | SUPORTE A LOJAS': 'SUPORTE LOJAS',
    'TI | SUPORTE A LOJAS (RETAGUARDA)' : 'SUPORTE A LOJAS',
    'TI | SUPORTE A LOJAS N2' : 'SUPORTE A LOJAS',
    'TI | SISTEMAS': 'SISTEMAS',
    'TI | SISTEMAS N2': 'SISTEMAS',
    'TI | PROJETOS': 'PROJETOS',
    'TI | E-COMMERCE' : 'E-COMMERCE',
    'TI | GOVERNANÇA' : 'GOVERNANÇA',
    'TI | INFRAESTRUTURA' : 'INFRAESTRUTURA',
    'TI | BUSINESS INTELLIGENCE N1' : 'BUSINESS INTELLIGENCE',
    'TI | LINE CARTUCHOS' : 'SUPORTE LOCAL',
    'TI | '
    'ADMINISTRAÇÃO DE PESSOAL': 'RH OPERAÇÕES',
    'REMUNERAÇÃO' : 'RH OPERAÇÕES',
    'CROWN IT - ATUALIZAÇÃO DE VERSÃO LINX' : 'SISTEMAS',
    'CROWN IT N2' : 'SISTEMAS',
    'DEPARTAMENTO FISCAL' : 'FISCAL',
    'MANUTENÇÃO HUB': 'MANUTENÇÃO',
    'MANUTENÇÃO LOJAS': 'MANUTENÇÃO',
    'GPP / SEGURANÇA (HUB)': 'INFRAESTRUTURA',
    'GPP / SEGURANÇA (LOJAS)': 'INFRAESTRUTURA'
}

In [12]:
# Criando os campos email_requerente e dominio_email
df = df.rename(columns={'requerente': 'email_requerente'})
df['dominio_email'] = df['email_requerente'].str.split('@').str[1]
df['dominio_email'] = df['dominio_email'].fillna('Não atribuído')
df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email
0,27675,gerente.tommyoutletparana@inbrands.com.br,Rescisão,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,MÉDIA,Rescisão,Rescisão > Rescisão > Análise De Desligamento,TH - OUTLET PARANA,NaN,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 15:00:09,2026-05-18 15:00:09,NaT,2026-05-26 08:00:09,NaT,NaT,No prazo,NaT,inbrands.com.br
1,27674,jenyffer.santana@inbrands.com.br,Problema Ao Iniciar O Excel,EM ATENDIMENTO (ATRIBUÍDO),Incidente,Formcreator,BAIXA,Onedrive,Onedrive > Outros Problemas Relacionados Ao Onedrive,CD EMBU,GABRIEL LIMA,TI | SUPORTE LOCAL,Não atribuído,2026-05-18 14:51:29,2026-05-18 14:59:17,2026-05-18 16:51:29,2026-05-20 11:51:29,NaT,NaT,No prazo,2026-05-18 02:00:00,inbrands.com.br
2,27673,ellus.iguatemijk@inbrands.com.br,Plano De Saúde,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,ALTA,Benefícios,Benefícios > Plano De Saúde > Inclusão Colaborador,ELLUS - JK IGUATEMI,ALESSANDRA RIBEIRO,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 14:49:20,2026-05-18 14:49:20,NaT,2026-05-21 10:49:20,NaT,NaT,No prazo,NaT,inbrands.com.br
3,27672,kelly.almeida@inbrands.com.br,Desagendamento Em Duplicidade Na Tela De Concilçiação Linx/Equals,NOVO,Requisição,Formcreator,BAIXA,Financeiro,Financeiro > Vendas Pendentes - Equals,HUB-SP - CENESP BL C,NaN,TI | SISTEMAS,Não atribuído,2026-05-18 14:44:20,2026-05-18 14:44:20,2026-05-19 13:44:20,2026-05-25 17:44:20,NaT,NaT,No prazo,2026-05-18 23:00:00,inbrands.com.br
4,27671,tommy.higienopolis@inbrands.com.br,Luz Da Loja Piscando,SOLUCIONADO,Incidente,Formcreator,BAIXA,Iluminação - Lojas,Iluminação - Lojas > Piscando,TH - HIGIENOPOLIS,NaN,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 14:41:24,2026-05-18 14:54:55,NaT,2026-06-29 14:41:24,2026-05-18 14:54:55,NaT,No prazo,NaT,inbrands.com.br
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27109,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br
27110,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br
27111,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15:37:26,2025-07-16 16:10:09,2025-04-08 18:37:26,2025-04-15 17:31:28,2025-04-10 18:41:44,2025-04-12 00:04:19,Vencido,2026-05-18 03:00:00,inbrands.com.br
27112,68,gerente.s27@inbrands.com.br,Estou Precisando Passar Venda No Linxpos E A Tela De Vendas Não Está Abrindo. Parou De Funcionar Hoje A Tarde Depois De Uma Atualização Feira Por Aí Mesmo.,FECHADO,Incidente,Chat,MÉDIA,Linxpos Manager,Linxpos Manager > Linxpos Manager Não Abre,SL27 - OSCAR FREIRE,GABRIEL MARIANO,TI | SUPORTE A LOJAS,00:00:00,2025-04-07 19:04:54,2025-04-10 12:04:27,NaT,None,2025-04-08 09:33:41,2025-04-10 12:04:27,Não atribuído,NaT,inbrands.com.br


In [13]:
inicio = df['data_criacao'].min()
fim = pd.Timestamp.today()

In [14]:
calendario = pd.date_range(
    start=inicio,
    end=fim,
    freq='D'
)

In [15]:
map_dias = {
    'Monday': 'SEGUNDA',
    'Tuesday': 'TERÇA',
    'Wednesday': 'QUARTA',
    'Thursday': 'QUINTA',
    'Friday': 'SEXTA',
    'Saturday': 'SÁBADO',
    'Sunday': 'DOMINGO'
}

In [ ]:
map_filial = {
    
}

In [48]:
df['localizacao'] = (
    df['localizacao']
    .astype(str)
    .str.upper()
    .str.strip()
)

df['codigo_unidade'] = (
    df['localizacao']
    .map(map_filial)
)

df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao,subcategoria,status_macro,tempo_primeira_resposta_horas,tempo_resolucao_horas,status_sla,sla_previsto_horas,flag_sla_dentro_prazo,faixa_sla,data_abertura_merge,data_fechamento_merge,flag_sem_tecnico
0,27675,gerente.tommyoutletparana@inbrands.com.br,Rescisão,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,MÉDIA,Rescisão,Rescisão > Rescisão > Análise De Desligamento,TH - OUTLET PARANA,,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 15:00:09,2026-05-18 15:00:09,NaT,2026-05-26 08:00:09,NaT,NaT,No prazo,NaT,inbrands.com.br,NaN,ADMINISTRAÇÃO DE PESSOAL,,Rescisão > Análise De Desligamento,Em atendimento,NaN,NaN,NÃO CLASSIFICADO,8.0,NaN,EM ANDAMENTO,2026-05-18,NaT,0
1,27674,jenyffer.santana@inbrands.com.br,Problema Ao Iniciar O Excel,EM ATENDIMENTO (ATRIBUÍDO),Incidente,Formcreator,BAIXA,Onedrive,Onedrive > Outros Problemas Relacionados Ao Onedrive,CD EMBU,GABRIEL LIMA,TI | SUPORTE LOCAL,Não atribuído,2026-05-18 14:51:29,2026-05-18 14:59:17,2026-05-18 16:51:29,2026-05-20 11:51:29,NaT,NaT,No prazo,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE LOCAL,Outros Problemas Relacionados Ao Onedrive,Em atendimento,-12.858056,NaN,NÃO CLASSIFICADO,24.0,NaN,EM ANDAMENTO,2026-05-18,NaT,0
2,27673,ellus.iguatemijk@inbrands.com.br,Plano De Saúde,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,ALTA,Benefícios,Benefícios > Plano De Saúde > Inclusão Colaborador,ELLUS - JK IGUATEMI,ALESSANDRA RIBEIRO,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 14:49:20,2026-05-18 14:49:20,NaT,2026-05-21 10:49:20,NaT,NaT,No prazo,NaT,inbrands.com.br,NaN,ADMINISTRAÇÃO DE PESSOAL,,Plano De Saúde > Inclusão Colaborador,Em atendimento,NaN,NaN,NÃO CLASSIFICADO,4.0,NaN,EM ANDAMENTO,2026-05-18,NaT,0
3,27672,kelly.almeida@inbrands.com.br,Desagendamento Em Duplicidade Na Tela De Concilçiação Linx/Equals,NOVO,Requisição,Formcreator,BAIXA,Financeiro,Financeiro > Vendas Pendentes - Equals,HUB-SP - CENESP BL C,,TI | SISTEMAS,Não atribuído,2026-05-18 14:44:20,2026-05-18 14:44:20,2026-05-19 13:44:20,2026-05-25 17:44:20,NaT,NaT,No prazo,2026-05-18 23:00:00,inbrands.com.br,NaN,TI,SISTEMAS,Vendas Pendentes - Equals,Aberto,8.261111,NaN,NÃO CLASSIFICADO,24.0,NaN,EM ANDAMENTO,2026-05-18,NaT,0
4,27671,tommy.higienopolis@inbrands.com.br,Luz Da Loja Piscando,SOLUCIONADO,Incidente,Formcreator,BAIXA,Iluminação - Lojas,Iluminação - Lojas > Piscando,TH - HIGIENOPOLIS,,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 14:41:24,2026-05-18 14:54:55,NaT,2026-06-29 14:41:24,2026-05-18 14:54:55,NaT,No prazo,NaT,inbrands.com.br,NaN,MANUTENÇÃO,LOJAS,Piscando,Fechado,NaN,0.225278,FINALIZADO,24.0,1.0,ATÉ 1H,2026-05-18,NaT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27109,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9703.641944,0.635000,FINALIZADO,24.0,1.0,ATÉ 1H,2025-04-08,2025-04-10,0
27110,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,NaN,TI

In [17]:
# Tratamento do campo nome tecnico e grupo

df['nome_tecnico'] = df['nome_tecnico'].astype(str)
df['grupo'] = df['grupo'].astype(str)


df[['departamento', 'grupo_descricao']] = (
    df['grupo']
    .str.split('|', n=1, expand=True)
)

df['nome_tecnico'] = df['nome_tecnico'].str.strip()
df['departamento'] = df['departamento'].str.strip()
df['grupo_descricao'] = df['grupo_descricao'].str.strip()

df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao
0,27675,gerente.tommyoutletparana@inbrands.com.br,Rescisão,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,MÉDIA,Rescisão,Rescisão > Rescisão > Análise De Desligamento,TH - OUTLET PARANA,NaN,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 15:00:09,2026-05-18 15:00:09,NaT,2026-05-26 08:00:09,NaT,NaT,No prazo,NaT,inbrands.com.br,NaN,ADMINISTRAÇÃO DE PESSOAL,NaN
1,27674,jenyffer.santana@inbrands.com.br,Problema Ao Iniciar O Excel,EM ATENDIMENTO (ATRIBUÍDO),Incidente,Formcreator,BAIXA,Onedrive,Onedrive > Outros Problemas Relacionados Ao Onedrive,CD EMBU,GABRIEL LIMA,TI | SUPORTE LOCAL,Não atribuído,2026-05-18 14:51:29,2026-05-18 14:59:17,2026-05-18 16:51:29,2026-05-20 11:51:29,NaT,NaT,No prazo,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE LOCAL
2,27673,ellus.iguatemijk@inbrands.com.br,Plano De Saúde,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,ALTA,Benefícios,Benefícios > Plano De Saúde > Inclusão Colaborador,ELLUS - JK IGUATEMI,ALESSANDRA RIBEIRO,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 14:49:20,2026-05-18 14:49:20,NaT,2026-05-21 10:49:20,NaT,NaT,No prazo,NaT,inbrands.com.br,NaN,ADMINISTRAÇÃO DE PESSOAL,NaN
3,27672,kelly.almeida@inbrands.com.br,Desagendamento Em Duplicidade Na Tela De Concilçiação Linx/Equals,NOVO,Requisição,Formcreator,BAIXA,Financeiro,Financeiro > Vendas Pendentes - Equals,HUB-SP - CENESP BL C,NaN,TI | SISTEMAS,Não atribuído,2026-05-18 14:44:20,2026-05-18 14:44:20,2026-05-19 13:44:20,2026-05-25 17:44:20,NaT,NaT,No prazo,2026-05-18 23:00:00,inbrands.com.br,NaN,TI,SISTEMAS
4,27671,tommy.higienopolis@inbrands.com.br,Luz Da Loja Piscando,SOLUCIONADO,Incidente,Formcreator,BAIXA,Iluminação - Lojas,Iluminação - Lojas > Piscando,TH - HIGIENOPOLIS,NaN,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 14:41:24,2026-05-18 14:54:55,NaT,2026-06-29 14:41:24,2026-05-18 14:54:55,NaT,No prazo,NaT,inbrands.com.br,NaN,MANUTENÇÃO,LOJAS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27109,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE A LOJAS
27110,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE A LOJAS
27111,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15:37:26,2025-07-16 16:10:09,2025-04-08 18:37:26,2025-04-15 17:31:28,2025-04-10 18:41:44,2025-04-12 00:04:19,Vencido,2026-05-18 03:00:00,inbrands.com.br,NaN,TI,SUPORTE A LOJAS
27112,68,gerente.s27@inbrands.com.br,Estou Precisando Passar Venda No Linxpos E A Tela De Vendas Não Está Abrindo. Parou De Funcionar Hoje A Tarde Depois De Uma Atualização Feira Por Aí Mesmo.,FECHADO,Incidente,Chat,MÉDIA,Linxpos Manager,Linxpos Manager > Linxpos Manager Não Abre,SL27 - OSCAR FREIRE,GABRIEL MARIANO,TI | SUPORTE A LOJAS,00:00:00,2025-04-07 19:04:54,2025-04-10 12:04:27,NaT,None,2025-04-08 09:33:41,2025-04-10 12:04:27,Não atribuído,NaT,inbrands.com

In [18]:
df = df.replace(pd.NaT, "")

# Tratamento do campo Categoria
df['categoria'] = df['categoria'].astype(str)

df[['categoria', 'subcategoria']] = (
    df['categoria_completa']
    .str.split('>', n=1, expand=True)
)

# Remover espaços extras e dados nulos
df['categoria'] = df['categoria'].str.strip()
df['subcategoria'] = df['subcategoria'].str.strip().fillna('Não atribuído')

df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao,subcategoria
0,27675,gerente.tommyoutletparana@inbrands.com.br,Rescisão,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,MÉDIA,Rescisão,Rescisão > Rescisão > Análise De Desligamento,TH - OUTLET PARANA,,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 15:00:09,2026-05-18 15:00:09,NaT,2026-05-26 08:00:09,NaT,NaT,No prazo,NaT,inbrands.com.br,,ADMINISTRAÇÃO DE PESSOAL,,Rescisão > Análise De Desligamento
1,27674,jenyffer.santana@inbrands.com.br,Problema Ao Iniciar O Excel,EM ATENDIMENTO (ATRIBUÍDO),Incidente,Formcreator,BAIXA,Onedrive,Onedrive > Outros Problemas Relacionados Ao Onedrive,CD EMBU,GABRIEL LIMA,TI | SUPORTE LOCAL,Não atribuído,2026-05-18 14:51:29,2026-05-18 14:59:17,2026-05-18 16:51:29,2026-05-20 11:51:29,NaT,NaT,No prazo,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE LOCAL,Outros Problemas Relacionados Ao Onedrive
2,27673,ellus.iguatemijk@inbrands.com.br,Plano De Saúde,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,ALTA,Benefícios,Benefícios > Plano De Saúde > Inclusão Colaborador,ELLUS - JK IGUATEMI,ALESSANDRA RIBEIRO,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 14:49:20,2026-05-18 14:49:20,NaT,2026-05-21 10:49:20,NaT,NaT,No prazo,NaT,inbrands.com.br,,ADMINISTRAÇÃO DE PESSOAL,,Plano De Saúde > Inclusão Colaborador
3,27672,kelly.almeida@inbrands.com.br,Desagendamento Em Duplicidade Na Tela De Concilçiação Linx/Equals,NOVO,Requisição,Formcreator,BAIXA,Financeiro,Financeiro > Vendas Pendentes - Equals,HUB-SP - CENESP BL C,,TI | SISTEMAS,Não atribuído,2026-05-18 14:44:20,2026-05-18 14:44:20,2026-05-19 13:44:20,2026-05-25 17:44:20,NaT,NaT,No prazo,2026-05-18 23:00:00,inbrands.com.br,,TI,SISTEMAS,Vendas Pendentes - Equals
4,27671,tommy.higienopolis@inbrands.com.br,Luz Da Loja Piscando,SOLUCIONADO,Incidente,Formcreator,BAIXA,Iluminação - Lojas,Iluminação - Lojas > Piscando,TH - HIGIENOPOLIS,,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 14:41:24,2026-05-18 14:54:55,NaT,2026-06-29 14:41:24,2026-05-18 14:54:55,NaT,No prazo,NaT,inbrands.com.br,,MANUTENÇÃO,LOJAS,Piscando
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27109,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais
27110,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais
27111,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15:37:26,2025-07-16 16:10:09,2025-04-08 18:37:26,2025-04-15 17:31:28,2025-04-10 18:41:44,2025-04-12 00:04:19,Vencido,2026-05-18 03:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Não atribuído
27112,68,gerente.s27@inbrands.com.br,Estou Precisando Passar Venda No Linxpos E A Tela De Vendas Não Está Abrindo. Parou De Funcionar Hoje A Tarde Depois De Uma Atualização Feira Por Aí Mesmo.,FECHADO,Incidente,Chat,MÉDIA,Linxpos Manager,Linxpos Manager > Li

In [19]:
# Criando a coluna de Status_macro

df['status'] = (df['status'].astype(str)
                .str.upper()
                .str.strip()
                )

map_status_macro = {'NOVO': 'Aberto',
                    'EM ATENDIMENTO (ATRIBUÍDO)': 'Em atendimento', 
                    'EM ATENDIMENTO (PLANEJADO)': 'Em atendimento',
                    'FECHADO': 'Fechado',
                    'SOLUCIONADO': 'Fechado',
                    'PENDENTE': 'Pendente',
                    }

df['status_macro'] = (df['status'].map(map_status_macro))

df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao,subcategoria,status_macro
0,27675,gerente.tommyoutletparana@inbrands.com.br,Rescisão,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,MÉDIA,Rescisão,Rescisão > Rescisão > Análise De Desligamento,TH - OUTLET PARANA,,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 15:00:09,2026-05-18 15:00:09,NaT,2026-05-26 08:00:09,NaT,NaT,No prazo,NaT,inbrands.com.br,,ADMINISTRAÇÃO DE PESSOAL,,Rescisão > Análise De Desligamento,Em atendimento
1,27674,jenyffer.santana@inbrands.com.br,Problema Ao Iniciar O Excel,EM ATENDIMENTO (ATRIBUÍDO),Incidente,Formcreator,BAIXA,Onedrive,Onedrive > Outros Problemas Relacionados Ao Onedrive,CD EMBU,GABRIEL LIMA,TI | SUPORTE LOCAL,Não atribuído,2026-05-18 14:51:29,2026-05-18 14:59:17,2026-05-18 16:51:29,2026-05-20 11:51:29,NaT,NaT,No prazo,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE LOCAL,Outros Problemas Relacionados Ao Onedrive,Em atendimento
2,27673,ellus.iguatemijk@inbrands.com.br,Plano De Saúde,EM ATENDIMENTO (ATRIBUÍDO),Requisição,Formcreator,ALTA,Benefícios,Benefícios > Plano De Saúde > Inclusão Colaborador,ELLUS - JK IGUATEMI,ALESSANDRA RIBEIRO,ADMINISTRAÇÃO DE PESSOAL,Não atribuído,2026-05-18 14:49:20,2026-05-18 14:49:20,NaT,2026-05-21 10:49:20,NaT,NaT,No prazo,NaT,inbrands.com.br,,ADMINISTRAÇÃO DE PESSOAL,,Plano De Saúde > Inclusão Colaborador,Em atendimento
3,27672,kelly.almeida@inbrands.com.br,Desagendamento Em Duplicidade Na Tela De Concilçiação Linx/Equals,NOVO,Requisição,Formcreator,BAIXA,Financeiro,Financeiro > Vendas Pendentes - Equals,HUB-SP - CENESP BL C,,TI | SISTEMAS,Não atribuído,2026-05-18 14:44:20,2026-05-18 14:44:20,2026-05-19 13:44:20,2026-05-25 17:44:20,NaT,NaT,No prazo,2026-05-18 23:00:00,inbrands.com.br,,TI,SISTEMAS,Vendas Pendentes - Equals,Aberto
4,27671,tommy.higienopolis@inbrands.com.br,Luz Da Loja Piscando,SOLUCIONADO,Incidente,Formcreator,BAIXA,Iluminação - Lojas,Iluminação - Lojas > Piscando,TH - HIGIENOPOLIS,,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 14:41:24,2026-05-18 14:54:55,NaT,2026-06-29 14:41:24,2026-05-18 14:54:55,NaT,No prazo,NaT,inbrands.com.br,,MANUTENÇÃO,LOJAS,Piscando,Fechado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27109,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado
27110,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado
27111,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15:37:26,2025-07-16 16:10:09,2025-04-08 18:37:26,2025-04-15 17:31:28,2025-04-10 18:41:44,2025-04-12 00:04:19,Vencido,2026-05-18 03:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Não atribuído,Fechado
27112,68,gerente.s27@inbrands.com.br,Estou Precisando Passar Venda No Linxpos E A Tela De Vendas Não Está Abrindo. Parou De Funcionar Hoje A Tarde Depois De 

## Métricas

In [20]:
### Tempo Primeira Resposta
df['tempo_primeira_resposta_horas'] = (
    (
        df['tempo_resposta']
        - df['data_criacao']
    )
    .dt.total_seconds()
    / 3600
)

In [21]:
### Tempo Resolução
df['tempo_resolucao_horas'] = (
    (
        df['data_solucao']
        - df['data_criacao']
    )
    .dt.total_seconds()
    / 3600
)

## SLA (Regra Corporativa)

In [22]:
condicoes = [

    df['data_solucao'].notnull(),

    (
        df['data_solucao'].isnull()
        &
        df['status'].isin([
            'ABERTO',
            'EM ANDAMENTO'
        ])
    ),

    (
        df['data_solucao'].isnull()
        &
        df['status'].str.contains(
            'PENDENTE',
            na=False
        )
    ),

    (
        df['nome_tecnico'].isnull()
    )
]

In [23]:
valores = [
    'FINALIZADO',
    'EM ABERTO',
    'PENDENTE',
    'NÃO ATRIBUÍDO'
]

In [24]:
df['status_sla'] = np.select(
    condicoes,
    valores,
    default='NÃO CLASSIFICADO'
)

In [25]:
map_sla = {
    'CRÍTICA': 2,
    'ALTA': 4,
    'MÉDIA': 8,
    'BAIXA': 24
}

In [26]:
df['sla_previsto_horas'] = (
    df['prioridade']
    .map(map_sla)
)

In [27]:
df['flag_sla_dentro_prazo'] = np.where(

    df['tempo_resolucao_horas'].isnull(),

    np.nan,

    np.where(
        df['tempo_resolucao_horas']
        <=
        df['sla_previsto_horas'],
        1,
        0
    )
)

In [28]:
def classificar_faixa_sla(tempo):

    if pd.isnull(tempo):
        return 'EM ANDAMENTO'

    elif tempo <= 1:
        return 'ATÉ 1H'

    elif tempo <= 4:
        return '1H A 4H'

    elif tempo <= 8:
        return '4H A 8H'

    elif tempo <= 18:
        return '8H A 18H'

    else:
        return 'ACIMA 18H'

In [29]:
df['faixa_sla'] = (
    df['tempo_resolucao_horas']
    .apply(classificar_faixa_sla)
)

## Criação das Tabelas de Dimensão

In [31]:
### 5.1 D_DATA
D_DATA = pd.DataFrame({
    'data_completa': calendario
})

D_DATA['data_merge'] = (
    D_DATA['data_completa']
    .dt.date
)

D_DATA['data_sk'] = (
    D_DATA['data_completa']
    .dt.strftime('%Y%m%d')
    .astype(int)
)

### Colunas Temporais ###

D_DATA['ano'] = (
    D_DATA['data_completa']
    .dt.year
)

D_DATA['mes'] = (
    D_DATA['data_completa']
    .dt.month
)

D_DATA['nome_mes'] = (
    D_DATA['data_completa']
    .dt.month_name()
)

D_DATA['trimestre'] = (
    D_DATA['data_completa']
    .dt.quarter
)

D_DATA['dia'] = (
    D_DATA['data_completa']
    .dt.day
)

D_DATA['dia_semana_num'] = (
    D_DATA['data_completa']
    .dt.dayofweek
)

D_DATA['dia_semana'] = (
    D_DATA['data_completa']
    .dt.day_name()
)

D_DATA['dia_semana'] = (
    D_DATA['dia_semana']
    .map(map_dias)
)

D_DATA['eh_dia_util'] = np.where(
    D_DATA['dia_semana_num'] < 5,
    1,
    0
)


In [32]:
### 5.2 D_REQUERENTE
D_REQUERENTE = (
        df[['email_requerente', 'dominio_email']]
        .drop_duplicates()
        .sort_values(by='email_requerente')
        .reset_index(drop=True)
)
D_REQUERENTE['requerente_sk'] = D_REQUERENTE.index + 1
D_REQUERENTE

,email_requerente,dominio_email,requerente_sk
0,,Não atribuído,1
1,Breno Maciel,Não atribuído,2
2,Felipe Gonçalves,Não atribuído,3
3,Gabriele Bispo,Não atribuído,4
4,Jefferson Barbosa,Não atribuído,5
...,...,...,...
969,yellen.moreira@inbrands.com.br,inbrands.com.br,970
970,ygor.parecy@inbrands.com.br,inbrands.com.br,971
971,yngrid.ferreira@inbrands.com.br,inbrands.com.br,972
972,yuki.sato@inbrands.com.br,inbrands.com.br,973


In [33]:
### 5.3 D_TECNICO
D_TECNICO = (
    df[
        df['nome_tecnico'].notnull()
    ][['nome_tecnico', 'departamento']]
    .drop_duplicates()
    .sort_values(by='nome_tecnico')
    .reset_index(drop=True)
)

# Criação da flag_sem_tecnico
df['flag_sem_tecnico'] = np.where(
    df['nome_tecnico'].isnull(),
    1,
    0
)

# Validar quantidade sem técnico
if df['flag_sem_tecnico'].sum() > 0:
    print("Sem Técnico")

df_sem_tecnico = df[
    df['flag_sem_tecnico'] == 1
]
D_TECNICO['tecnico_sk'] = D_TECNICO.index + 1
D_TECNICO

,nome_tecnico,departamento,tecnico_sk
0,,ADMINISTRAÇÃO DE PESSOAL,1
1,,CROWN IT - ATUALIZAÇÃO DE VERSÃO LINX,2
2,,TI,3
3,,MANUTENÇÃO,4
4,,GPP / SEGURANÇA (HUB),5
...,...,...,...
63,VALDIR BOSCO DA SILVA JUNIOR,,64
64,VANUSA PEREIRA SALES,ADMINISTRAÇÃO DE PESSOAL,65
65,WESLLEY DE SANTANA MOREIRA,TI,66
66,WESLLEY DE SANTANA MOREIRA,CROWN IT N2,67


In [71]:
### 5.4 D_LOCALIZACAO
D_LOCALIZACAO = (
    df[['localizacao']]
    .drop_duplicates()
    .reset_index(drop=True)
)

D_LOCALIZACAO['localizacao'] = (

    D_LOCALIZACAO['localizacao']
    .astype(str)
    .str.upper()
    .str.strip()
)

D_LOCALIZACAO['localizacao'] = np.where(

    D_LOCALIZACAO['localizacao'].isin(
        valores_invalidos
    ),

    np.nan,

    D_LOCALIZACAO['localizacao']
)

D_LOCALIZACAO['marca'] = (
    D_LOCALIZACAO['localizacao']
    .apply(extrair_marca)
)

D_LOCALIZACAO['tipo_operacao'] = (

    D_LOCALIZACAO['localizacao']
    .apply(classificar_tipo_operacao)
)

D_LOCALIZACAO['unidade_padronizada'] = (

    D_LOCALIZACAO['localizacao']
    .apply(limpar_unidade)
)

D_LOCALIZACAO['cidade'] = (

    D_LOCALIZACAO['unidade_padronizada']
    .map(map_cidade)
)

D_LOCALIZACAO['estado'] = (

    D_LOCALIZACAO['cidade']
    .map(map_estado)
)

D_LOCALIZACAO['regiao'] = (

    D_LOCALIZACAO['estado']
    .map(map_regiao)
)

D_LOCALIZACAO['flag_outlet'] = np.where(

    D_LOCALIZACAO['localizacao']
    .str.contains(
        'OUTLET',
        na=False
    ),

    1,

    0
)

D_LOCALIZACAO['flag_hub'] = np.where(

    D_LOCALIZACAO['localizacao']
    .str.contains(
        'HUB',
        na=False
    ),

    1,

    0
)

D_LOCALIZACAO['flag_cd'] = np.where(

    D_LOCALIZACAO['localizacao']
    .str.startswith(
        'CD',
        na=False
    ),

    1,

    0
)

D_LOCALIZACAO['flag_corporativo'] = np.where(

    D_LOCALIZACAO['localizacao']
    .str.contains(
        'CORPORATIVO',
        na=False
    ),

    1,

    0
)

D_LOCALIZACAO = (

    D_LOCALIZACAO
    .sort_values(
        by='unidade_padronizada'
    )
    .reset_index(drop=True)
)

D_LOCALIZACAO['localizacao_sk'] = (

    D_LOCALIZACAO.index + 1
)

In [34]:
### 5.5 D_CATEGORIA
D_CATEGORIA = (
    df[['categoria', 'subcategoria']]
    .drop_duplicates()
    .sort_values(by='categoria')
    .reset_index(drop=True)
)
D_CATEGORIA['categoria_sk'] = D_CATEGORIA.index + 1
D_CATEGORIA

,categoria,subcategoria,categoria_sk
0,Abertura Via Chat Indevida,Não atribuído,1
1,Acessos,Liberação De Pastas Da Rede,2
2,Acessos,Acesso Ao Ts,3
3,Acessos,Não atribuído,4
4,Acessos,Liberação De Acesso Linx,5
...,...,...,...
514,Verdanadesk,Criação De Filtros / Regras,515
515,Verdanadesk,Ajustes,516
516,Verdanadesk,Criação De Categoria,517
517,Verdanadesk,Inclusão De Usuário,518


In [35]:
### 5.6 D_GRUPO

D_GRUPO = (
    df[['grupo']]
    .drop_duplicates()
)

D_GRUPO['grupo'] = (
    D_GRUPO['grupo']
    .astype(str)
    .str.upper()
    .str.strip()
)

D_GRUPO['area_responsavel'] = (
    D_GRUPO['grupo']
    .map(map_area)
)

D_GRUPO['torre_servico'] = (
    D_GRUPO['grupo']
    .map(map_torre)
)

D_GRUPO = (
    D_GRUPO
    .sort_values(by='grupo')
    .reset_index(drop=True)
)

D_GRUPO['grupo_sk'] = (
    D_GRUPO.index + 1
)


In [36]:
### 5.7 D_STATUS
D_STATUS = (
    df[['status', 'status_macro']]
    .drop_duplicates()
    .sort_values(by='status')
    .reset_index(drop=True)
)
D_STATUS['status_sk'] = D_STATUS.index + 1
D_STATUS

,status,status_macro,status_sk
0,EM ATENDIMENTO (ATRIBUÍDO),Em atendimento,1
1,EM ATENDIMENTO (PLANEJADO),Em atendimento,2
2,FECHADO,Fechado,3
3,NOVO,Aberto,4
4,PENDENTE,Pendente,5
5,SOLUCIONADO,Fechado,6


In [37]:
### 5.8 D_TIPO
D_TIPO = (
    df[['tipo']]
    .drop_duplicates()
    .sort_values(by='tipo')
    .reset_index(drop=True)
)
D_TIPO['tipo_sk'] = D_TIPO.index + 1
D_TIPO

,tipo,tipo_sk
0,Incidente,1
1,Requisição,2


In [38]:
### 5.9 D_ORIGEM_ABERTURA
D_ORIGEM_ABERTURA = (
    df[['origem_abertura']]
    .drop_duplicates()
    .sort_values(by='origem_abertura')
    .reset_index(drop=True)
)
D_ORIGEM_ABERTURA['origem_abertura_sk'] = D_ORIGEM_ABERTURA.index + 1
D_ORIGEM_ABERTURA

,origem_abertura,origem_abertura_sk
0,Chat,1
1,Direct,2
2,E-Mail,3
3,Formcreator,4
4,Helpdesk,5
5,Other,6


In [39]:
### 5.10 D_PRIORIDADE
D_PRIORIDADE = (
    df[['prioridade']]
    .drop_duplicates()
    .sort_values(by='prioridade')
    .reset_index(drop=True)
)
D_PRIORIDADE['prioridade_sk'] = D_PRIORIDADE.index + 1
D_PRIORIDADE

,prioridade,prioridade_sk
0,ALTA,1
1,BAIXA,2
2,CRÍTICA,3
3,MUITO ALTA,4
4,MUITO BAIXA,5
5,MÉDIA,6


In [40]:
### 5.11 D_SLA
D_SLA = (
    df[
        [
            'sla_previsto_horas',
            'status_sla',
            'faixa_sla'
        ]
    ]
    .drop_duplicates()
    .sort_values(
        by=['sla_previsto_horas']
    )
    .reset_index(drop=True)
)
D_SLA['sla_sk'] = (
    D_SLA.index + 1
)

## Criação da Tabela Fato 

In [41]:
F_CHAMADO = df.merge(

    D_DATA[
        [
            'data_sk',
            'data_merge'
        ]
    ],

    left_on='data_abertura_merge',
    right_on='data_merge',

    how='left'
)

F_CHAMADO = F_CHAMADO.rename(
    columns={
        'data_sk': 'data_abertura_sk'
    }
)

In [42]:
F_CHAMADO = F_CHAMADO.merge(

    D_DATA[
        [
            'data_sk',
            'data_merge'
        ]
    ],

    left_on='data_fechamento_merge',
    right_on='data_merge',

    how='left'
)

In [43]:
F_CHAMADO = F_CHAMADO.rename(
    columns={
        'data_sk': 'data_fechamento_sk'
    }
)

In [72]:
# Criação da Tabela fato

F_CHAMADOS = df.merge(
    D_TECNICO,
    how='left',
    left_on=['nome_tecnico', 'departamento'],
    right_on=['nome_tecnico', 'departamento']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_CATEGORIA,
    how='left',
    left_on=['categoria', 'subcategoria'],
    right_on=['categoria', 'subcategoria']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_STATUS,
    how='left',
    left_on=['status','status_macro'],
    right_on=['status','status_macro']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_REQUERENTE,
    how='left',
    left_on=['email_requerente', 'dominio_email'],
    right_on=['email_requerente', 'dominio_email']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_TIPO,
    how='left',
    left_on=['tipo'],
    right_on=['tipo']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_ORIGEM_ABERTURA,
    how='left',
    left_on=['origem_abertura'],
    right_on=['origem_abertura']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_GRUPO,
    how='left',
    left_on=['grupo'],
    right_on=['grupo']
)

F_CHAMADOS = F_CHAMADOS.merge(

    D_SLA[
        [
            'sla_sk',
            'sla_previsto_horas',
            'status_sla',
            'faixa_sla'
        ]
    ],

    on=[
        'sla_previsto_horas',
        'status_sla',
        'faixa_sla'
    ],

    how='left'
)

F_CHAMADOS = F_CHAMADOS.merge(

    D_DATA[
        [
            'data_sk',
            'data_merge'
        ]
    ],

    left_on='data_fechamento_merge',
    right_on='data_merge',

    how='left'
)

F_CHAMADO = F_CHAMADO.merge(

    D_LOCALIZACAO[
        [
            'localizacao_sk',
            'localizacao'
        ]
    ],

    on='localizacao',

    how='left'
)

F_CHAMADOS = F_CHAMADOS.drop(columns=
                             ['nome_tecnico', 
                              'grupo',
                              'departamento', 
                              'categoria', 
                              'categoria', 
                              'subcategoria',   
                              'status',
                              'email_requerente',
                              'tipo',
                              'origem_abertura',
                              'prioridade',
                              'localizacao',
                              'categoria_completa',
                              'tempo_chamado',
                              'data_criacao',
                              'ultima_atualizacao',
                              'tempo_atribuicao',
                              'tempo_solucao',
                              'data_fechamento',
                              'data_solucao',
                              'expirado',
                              'tempo_resposta',
                              'grupo_descricao',
                              'dominio_email',
                              'status_macro',
                              'sla_previsto_horas',
                              'tempo_primeira_resposta_horas',
                              'codigo_unidade',
                              'tempo_resolucao_horas',
                              'status_sla',
                              'faixa_sla',
                              'data_abertura_merge',
                              'data_fechamento_merge',
                              'area_responsavel',
                              'torre_servico',
                              'data_merge',
                              'data_sk']
                              )
F_CHAMADOS = F_CHAMADOS.rename(columns={'id': 'id_chamado'})
F_CHAMADOS['chamado_sk'] = F_CHAMADOS.index + 1
F_CHAMADOS

,id_chamado,titulo,flag_sla_dentro_prazo,flag_sem_tecnico,tecnico_sk,categoria_sk,status_sk,requerente_sk,tipo_sk,origem_abertura_sk,grupo_sk,sla_sk,chamado_sk
0,27675,Rescisão,NaN,0,1,459,1,414,2,4,2,14,1
1,27674,Problema Ao Iniciar O Excel,NaN,0,29,378,1,515,1,4,23,20,2
2,27673,Plano De Saúde,NaN,0,10,49,1,203,2,4,2,2,3
3,27672,Desagendamento Em Duplicidade Na Tela De Concilçiação Linx/Equals,NaN,0,3,214,4,563,2,4,18,20,4
4,27671,Luz Da Loja Piscando,1.0,0,4,267,6,874,1,4,9,16,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27109,71,Notas Fiscais Não Aparecem Para Entrada.,1.0,0,61,224,3,820,2,4,20,16,27110
27110,70,Nota Fiscal De Doação,1.0,0,61,224,3,820,2,4,20,19,27111
27111,69,Peça Não Cadastrada,0.0,0,61,363,3,821,2,4,20,25,27112
27112,68,Estou Precisando Passar Venda No Linxpos E A Tela De Vendas Não Está Abrindo. Parou De Funcionar Hoje A Tarde Depois De Uma Atualização Feira Por Aí Mesmo.,0.0,0,32,329,3,392,1,1,20,11,27113


In [45]:
df.columns = df.columns.str.strip()
df.columns

Index(['id', 'email_requerente', 'titulo', 'status', 'tipo', 'origem_abertura',
       'prioridade', 'categoria', 'categoria_completa', 'localizacao',
       'nome_tecnico', 'grupo', 'tempo_chamado', 'data_criacao',
       'ultima_atualizacao', 'tempo_atribuicao', 'tempo_solucao',
       'data_solucao', 'data_fechamento', 'expirado', 'tempo_resposta',
       'dominio_email', 'codigo_unidade', 'departamento', 'grupo_descricao',
       'subcategoria', 'status_macro', 'tempo_primeira_resposta_horas',
       'tempo_resolucao_horas', 'status_sla', 'sla_previsto_horas',
       'flag_sla_dentro_prazo', 'faixa_sla', 'data_abertura_merge',
       'data_fechamento_merge', 'flag_sem_tecnico'],
      dtype='str')

## Exportação via Excel

In [ ]:
# Criar arquivos excel

with pd.ExcelWriter("Tratamento_chamados.xlsx") as writer:
    D_DATA.to_excel(writer, sheet_name="D_DATA", index=False)
    D_REQUERENTE.to_excel(writer, sheet_name="D_REQUERENTE", index=False)
    D_TECNICO.to_excel(writer, sheet_name="D_TECNICO", index=False)
    D_LOCALIZACAO.to_excel(writer, sheet_name="D_LOCALIZACAO", index=False)
    D_CATEGORIA.to_excel(writer, sheet_name="D_CATEGORIA", index=False)
    D_GRUPO.to_excel(writer, sheet_name="D_GRUPO", index=False)
    D_STATUS.to_excel(writer, sheet_name="D_STATUS", index=False)
    D_TIPO.to_excel(writer, sheet_name="D_TIPO", index=False)
    D_ORIGEM_ABERTURA.to_excel(writer, sheet_name="D_ORIGEM_ABERTURA", index=False)
    D_PRIORIDADE.to_excel(writer, sheet_name="D_PRIORIDADE", index=False)
    D_SLA.to_excel(writer, sheet_name="D_SLA", index=False)
    F_CHAMADOS.to_excel(writer, sheet_name="F_CHAMADOS", index=False)